In [2]:
!pip install torch torchvision torchaudio
!pip install diffusers transformers accelerate peft
!pip install   pillow numpy opencv-python datasets
!pip install git+https://github.com/huggingface/diffusers
!pip install streamlit

# Required libraries for project.

  Cloning https://github.com/huggingface/diffusers to /private/var/folders/90/7bhc4qw17dd0tc8r6137_d040000gn/T/pip-req-build-gnsnp5gb
  Running command git clone --filter=blob:none --quiet https://github.com/huggingface/diffusers /private/var/folders/90/7bhc4qw17dd0tc8r6137_d040000gn/T/pip-req-build-gnsnp5gb
  Resolved https://github.com/huggingface/diffusers to commit 20364fe5a2afbddbf2ec5097ec5d483ce0cc689a
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [36]:
# !pip install huggingface_hub


from huggingface_hub import snapshot_download
# from hugging face download the dataset :
# dataset is Disney video generation dataset.
# snapshot_download() downloads an entire repository at a given revision. It uses internally hf_hub_download() which means all downloaded files are also cached on your local disk. 

snapshot_download(
    repo_id="Wild-Heart/Disney-VideoGeneration-Dataset",
    repo_type="dataset",
    local_dir="./video_dataset"
)

Fetching 75 files: 100%|██████████| 75/75 [00:00<00:00, 600.89it/s]


'/Users/pagupta/Desktop/GD_Projects/module_2_all_courses/ML_Specialization_coursera/video_dataset'

In [37]:
# import the processor class for BLIP

from transformers import BlipProcessor, BlipForConditionalGeneration
#Pillow(PIL) :  Python library for image processing, allowing users to open, manipulate, and save various file formats 
from PIL import Image
# BLIP model runs using torch tensors.
import torch

# BLIP (Bootstrapped Language-Image Pretraining) is a vision-language pretraining (VLP) framework designed 
# for both understanding and generation tasks. Most existing pretrained models are only good at one or the other. 
# It uses a captioner to generate captions and a filter to remove the noisy captions. This increases training data quality and 
# more effectively uses the messy web data.


# import the processor class for BLIP
# this handle image preprocessing + test tokenization automatically.
processor = BlipProcessor.from_pretrained(
    "Salesforce/blip-image-captioning-base"
)
# "Salesforce/blip-image-captioning-base" is model id. 
# BLIP model runs using torch tensors.


# BlipForConditionalGeneration. is used because image captioning is a sequence generation task.
# it generate text (caption) conditioned on an image.

model = BlipForConditionalGeneration.from_pretrained(
    "Salesforce/blip-image-captioning-base"
).to("mps")


def generate_caption(image):
    # returns pytorch tensors. for image.
    inputs = processor(image, return_tensors="pt").to("mps")


    # autoregressive text generation.
    # inputs is the input text arguments.
    #model predict the next token.
    # until it generates a complete caption.
    out = model.generate(**inputs)


    # convert token IDs back to readable text.
    # skip_special tokens.
    return processor.decode(out[0], skip_special_tokens=True)

Loading weights: 100%|██████████| 473/473 [00:00<00:00, 567.71it/s, Materializing param=vision_model.post_layernorm.weight]
The tied weights mapping and config for this model specifies to tie text_decoder.cls.predictions.bias to text_decoder.cls.predictions.decoder.bias, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie text_decoder.bert.embeddings.word_embeddings.weight to text_decoder.cls.predictions.decoder.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
BlipForConditionalGeneration LOAD REPORT from: Salesforce/blip-image-captioning-base
Key                                       | Status     |  | 
------------------------------------------+------------+--+-
text_decoder.bert.embeddings.position_ids | UNEXP

In [38]:
# opencv is used to read video files frame by frame.
import cv2
import torch
import numpy as np
import torch.nn.functional as F
from torch.utils.data import Dataset



class VideoDataset(Dataset):
    def __init__(self, video_paths, captions, size=256, frames=16):
        self.video_paths = video_paths
        self.captions = captions
        self.size = size
        self.frames = frames

    def __len__(self):
        return len(self.video_paths)

    def load_video(self, path):
        # create a video capture object that allows frame-by-frame reading.
        cap = cv2.VideoCapture(path)
        total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

        # sample frame indices uniformly
        indices = np.linspace(0, total - 1, self.frames).astype(int)

        frames = []
        # current_frame track position in video
        current_frame = 0
        # target_id tracks which sampled index we want next.
        target_id = 0

        while cap.isOpened():
            ret, frame = cap.read()
            # ret is for whether reading is succeeded or not
            # frame is an image array.
            if not ret:
                break
            # if reading is not succeeded then just return
            #otherwise read the frame and append this frame to the frames array.

            if target_id < len(indices) and current_frame == indices[target_id]:
                frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                frames.append(frame)
                target_id += 1

            current_frame += 1

        cap.release() 
        # free the memory for next frame.

        

        
        if len(frames) == 0:
            raise ValueError(f"Failed to load any frames from video: {path}. check the path please.")

        

        # convert list of frames to numpy array. 
        # normalized the values and scaled then between 0 to 1.
        frames = np.array(frames).astype(np.float32) / 255.0

        # original shape we get is (frames,height,width,channels)
        # but  deep learning expects (frames,chanels,height,width)
        # so we reorder the each frame in frames list.
        frames = torch.from_numpy(frames).permute(0, 3, 1, 2)



        # interpolation used for resizing input tensors.
        # what is the input size, resize the input into required output size and bilinear mode used for smooth resizing.

        frames = F.interpolate(
            frames,
            size=(self.size, self.size),
            mode="bilinear",
            align_corners=False
        )

        return frames

    def __getitem__(self, idx):

        frames = self.load_video(self.video_paths[idx])
    # hugging face training format.
        return {
            "pixel_values": frames,
            "caption": self.captions[idx]
        }

In [39]:

with open("./video_dataset/videos.txt",'r') as f:
    # Use .strip() to remove the \n, and remove the extra 'videos/' 
    # since it's already in the text file
    videos = [f"./video_dataset/{x.strip()}" for x in f]

with open("./video_dataset/prompt.txt") as f:
    captions = [x.strip() for x in f]

# for caption in captions:
#     print(caption,end='\n')

In [ ]:
# del pipe
# import gc
# gc.collect()

# import torch
# torch.mps.empty_cache()

# powerful image-to-video generation model that can generate 2-4 second high resolution  videos conditioned on an input image.
from diffusers import StableVideoDiffusionPipeline

import torch
# from_pretrained  downloads pretrained stable video diffusion weights from huggingFace.
# dtype=fp16 reduces memory usage.



# The Pipeline Workflow (Inference)
# Input Image Encoding: The input image is converted into a latent representation using a Variational Autoencoder (VAE) encoder.

# Noise Augmentation: The encoded image is slightly augmented with noise (noise_aug_strength) to introduce stochasticity, allowing the model to generate variation.

# Denoising (UNet): The 3D UNet (now with temporal layers) predicts the noise in the video frames, iteratively denoising the latent representation over a series of steps.

# Temporal Consistency: The model ensures that objects do not change shape or color unexpectedly between frames.

# Decoding: The denoised latents are passed through the VAE decoder to create the final video frames.
pipe = StableVideoDiffusionPipeline.from_pretrained(
    "stabilityai/stable-video-diffusion-img2vid-xt",
    torch_dtype=torch.float16
)
# move entire model to Apple GPU
pipe.to("mps")

#DDPM(Denoising Diffusion Probabilistic Models)  refers to the discrete denoising scheduler from the paper as well as pipeline.
# iteratively removes noise from a Gaussian noise tensor over a set number of steps, guided by a UNet model, to reconstruct a clean image from random noise.
from diffusers import DDPMScheduler

# Swap the inference scheduler for a training-friendly DDPM scheduler
# the scheduler that comes with the pretrained Stable Video Diffusion pipeline is optimized for inference (generation), not training.
pipe.scheduler = DDPMScheduler.from_config(pipe.scheduler.config)

# 1. Setup - Accelerator handle the precision!
device = "mps"

 
# The pipeline contains multiple models:
# VAE → encodes/decodes images to latent space
# UNet → predicts noise during diffusion
# CLIP Image Encoder → encodes conditioning image
# Scheduler → controls diffusion steps
#  Image Processor


Loading pipeline components...: 100%|██████████| 5/5 [00:18<00:00,  3.69s/it]


In [47]:
# Peft _> parameter efficient fine tunning.   instead of training the entire huge model we traines only a small low rank adapters.
# this saves memory.
# this speeds up the training process
# prevents overfitting
# works great on small Gpu

# Lora: low rank adapter
from peft import LoraConfig, get_peft_model

# we are using here lora with rank = 8 
# lora replace weight update -> w=w+BA
# where A is rxd
# B is dxr

# lora alpha 
# final lora update:
# w=w+alpha*(BA)/r

# to_q=Query projection
# to_key=key prrojection
# value projection=>to_v
# output projection=>to_out

# initailize the lora weights from guassian distribution.

lora_config = LoraConfig(
    r=4,
    lora_alpha=8,
    target_modules=["to_q","to_k","to_v","to_out.0"],
    init_lora_weights="gaussian"
)

# apply lora to unet.
# unet is a convolutional neural netowrk(CNN) architecture designed for fast,precise image segmentation.
# it utilizes U-shaped structure consisting of a contracting path to capture context and a symmetric exapnding path for  precise localization connect by skip connections that transfer high-resolution features.

# takes existing UNet.
# injects Lora layers into specified modules.
# freezes original weights
# makes only lora weights trainable.


pipe.unet = get_peft_model(pipe.unet, lora_config)



In [44]:
pipe.unet.enable_gradient_checkpointing()
# forward pass compute activations
# activations are stored in memory
# backward pass uses stored activations to compute gradients.

pipe.enable_model_cpu_offload()
# loading entire model on GPU


In [49]:
# data loader is used to batch and shuffle dataset samples.
# Auto tokenizer converts text captions into token IDS
# accelerate simplifies mixed precision and device handling.

from torch.utils.data import DataLoader
from transformers import AutoTokenizer
from accelerate import Accelerator

dataset = VideoDataset(videos, captions, size=256, frames=16)
# create a dataset using vides and captions.


loader = DataLoader(
    dataset,
    batch_size=1,
    shuffle=True
)

accelerator = Accelerator(
    mixed_precision="fp16"
)
# AdamW optimizer is used for transformer based models.
# lr = learning rate.
# use unet parameters.
optimizer = torch.optim.AdamW(
    pipe.unet.parameters(),
    lr=1e-4
)

pipe.unet, optimizer, loader = accelerator.prepare(
    pipe.unet,
    optimizer,
    loader
)

In [ ]:

import torch
import torch.nn.functional as F
#maximum training step.
# global counter step to stop training 
num_steps = 1000
global_step = 0

device = "mps"

# vae -> variational encoder are probabilistic generative deep learning models that encode input data into a continous , structured latent space to reconstruct , denoise, or generate new similiar data.

# Video frames
#       ↓
# Convert frames → latent representation (VAE)
#       ↓
# Add random noise
#       ↓
# UNet predicts that noise
#       ↓
# Compute loss (MSE)
#       ↓
# Update model weights

pipe.vae.to(device)
pipe.image_encoder.to(device)

# freeze the VAE weights.
# freeze image encoder weights.

pipe.vae.requires_grad_(False) 
pipe.image_encoder.requires_grad_(False)

for epoch in range(2):
    for batch in loader:
        with accelerator.accumulate(pipe.unet):
            
            # pixel_values shape: [bs, frames, channels, height, width]
            # Assuming values are [0, 1]. SVD VAE expects [-1, 1].so we normalized the data.

            pixel_values = batch["pixel_values"].to(device=device, dtype=torch.float16)
            pixel_values_scaled = pixel_values * 2.0 - 1.0 
            
            bs, num_frames, c, h, w = pixel_values.shape

            # 1.Extract First Frame for Conditioning
            first_frames = pixel_values_scaled[:, 0, :, :, :] # [bs, c, h, w]

            # 2.Get Image Embeddings (SVD requires this instead of text)
            # Resize for CLIP Vision model (224x224)
            #CLIP image encoder expects 224×224 images.
            # diffusion models operate in latent space instead of pixel space.
            image_for_clip = F.interpolate(first_frames, size=(224, 224), mode='bilinear')
            with torch.no_grad():
                image_embeddings = pipe.image_encoder(image_for_clip).image_embeds
                # SVD expects sequence length of 1 for the embedding
                #image embedding (1024-dim vector)
                image_embeddings = image_embeddings.unsqueeze(1) # [bs, 1, 1024]


            # Encode the full video frames → video latents (what the model must denoise)
            # Encode the first frame → conditioning latents (what guides the video)

            # Encode Video with VAE 
            pixel_values_flat = pixel_values_scaled.view(bs * num_frames, c, h, w)
            with torch.no_grad():
                latents = pipe.vae.encode(pixel_values_flat).latent_dist.sample()
            latents = latents * pipe.vae.config.scaling_factor
            latents = latents.view(bs, num_frames, 4, h // 8, w // 8)
            # 512 x 512. -> 64 x 64  ~ 50% cheaper.
            # noisy_video_latent → predict noise

            # Encode First Frame Latents (Image Conditioning) 
            #We encode the first frame again and repeat it for every frame.
            # It is the conditioning signal that tells the model:
            # what the scene looks like
            with torch.no_grad():
                cond_latents = pipe.vae.encode(first_frames).latent_dist.sample()
            cond_latents = cond_latents * pipe.vae.config.scaling_factor
            # Repeat the single condition frame to match the number of video frames
            cond_latents = cond_latents.unsqueeze(1).repeat(1, num_frames, 1, 1, 1)

            #  Sample Noise 
            noise = torch.randn_like(latents)
            timesteps = torch.randint(
                0, pipe.scheduler.config.num_train_timesteps, (bs,), device=device
            ).long()

            noisy_latents = pipe.scheduler.add_noise(latents, noise, timesteps)

            # SVD UNet takes 8 channels: 4 (noisy video latents) + 4 (clean first-frame latents)
            model_input = torch.cat([noisy_latents, cond_latents], dim=2) 

            # Prepare Time IDs 
            # SVD specific time ids: [fps, motion_bucket_id, noise_aug_level]
            added_time_ids = torch.tensor(
                [[7.0, 127.0, 0.02]], 
                device=device
            ).repeat(bs, 1)

            # Forward pass 
            model_pred = pipe.unet(
                model_input,
                timesteps,
                encoder_hidden_states=image_embeddings,
                added_time_ids=added_time_ids
            ).sample

            # Loss 
            loss = torch.nn.functional.mse_loss(model_pred.float(), noise.float(), reduction="mean")

            accelerator.backward(loss)
            optimizer.step()
            optimizer.zero_grad()

            global_step += 1

            if global_step % 10 == 0:      
                print(f"Step: {global_step} | Loss: {loss.item():.4f}")

            if global_step >= num_steps:
                break
    if global_step >=num_steps:
        break

In [12]:
# pipe.unet.requires_grad_(False)

# for name, param in pipe.unet.named_parameters():
#     if "lora" in name:
#         param.requires_grad = True


pipe.unet.save_pretrained("./svd_video_lora")

In [66]:
from diffusers import StableVideoDiffusionPipeline
from peft import PeftModel

pipe = StableVideoDiffusionPipeline.from_pretrained(
    "stabilityai/stable-video-diffusion-img2vid-xt",
    torch_dtype=torch.float16
).to("mps")

pipe.unet = PeftModel.from_pretrained(
    pipe.unet,
    "./svd_video_lora"
)

Loading pipeline components...: 100%|██████████| 5/5 [00:18<00:00,  3.61s/it]


In [57]:

from PIL import Image

# 1. Provide a starting image
# Option A: Load an image from your dataset
# init_image = Image.open("path_to_your_disney_character_image.jpg").resize((256, 256))

# Option B: Create a dummy solid-color image just to test the pipeline runs
init_image = Image.new("RGB", (256, 256), color=(50, 50, 150))

# 2. Pass the 'image' argument instead of 'prompt'
frames = pipe(
    image=init_image,
    num_frames=16,
    height=256,
    width=256,
    decode_chunk_size=8, # Pro-tip: Add this to save VRAM during decoding!
).frames[0]


100%|██████████| 25/25 [3:13:15<00:00, 463.83s/it]   


In [ ]:


from IPython.display import Video, display
import torch
import gc
from diffusers import StableDiffusionPipeline
from diffusers.utils import export_to_video


# Load Text-to-Image model

t2i_pipe = StableDiffusionPipeline.from_pretrained(
    "runwayml/stable-diffusion-v1-5",
    torch_dtype=torch.float16
)



def generate_video(prompt):

    print("Generating initial image")

    
    t2i_pipe.to("mps")

    init_image = t2i_pipe(prompt).images[0]
    init_image = init_image.resize((256, 256))

    # Free GPU memory
    t2i_pipe.to("cpu")
    gc.collect()
    torch.mps.empty_cache()

    print("Generating video frames")

    
    pipe.to("mps")

    video_frames = pipe(
        image=init_image,
        num_frames=16,
        height=256,
        width=256,
        decode_chunk_size=4  
    ).frames[0]

#     User Prompt
#       ↓
#   Stable Diffusion (t2i_pipe)
#       ↓
#   Initial Image
#       ↓
#    Stable Video Diffusion (pipe)
#       ↓
#   Generated Video

    # Free GPU again
    pipe.to("cpu")
    gc.collect()
    torch.mps.empty_cache()

    print("Saving video")

    import imageio
    import numpy as np

    output_path = "gradio_output.mp4"

    frames_np = [np.array(frame).astype(np.uint8) for frame in video_frames]

    with imageio.get_writer(output_path, fps=7) as writer:
        for frame in frames_np:
            writer.append_data(frame)
    imageio.mimsave(output_path, frames_np, fps=8)

    return output_path




import gradio as gr

def generate(prompt):
    output_path = generate_video(prompt)
    return output_path

demo = gr.Interface(
    fn=generate,
    inputs=gr.Textbox(
        label="Enter Prompt",
        placeholder="cinematic disney style character walking in forest"
    ),
    outputs=gr.Video(label="Generated Video"),
    title="Text to Video Generator",
    description="Generate videos using Stable Diffusion + Stable Video Diffusion"
)

demo.launch()

Loading weights: 100%|██████████| 396/396 [00:01<00:00, 216.03it/s, Materializing param=visual_projection.weight]
StableDiffusionSafetyChecker LOAD REPORT from: /Users/pagupta/.cache/huggingface/hub/models--runwayml--stable-diffusion-v1-5/snapshots/451f4fe16113bff5a5d2269ed5ad43b0592e9a14/safety_checker
Key                                               | Status     |  | 
--------------------------------------------------+------------+--+-
vision_model.vision_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 196/196 [00:01<00:00, 179.19it/s, Materializing param=text_model.final_layer_norm.weight]
CLIPTextModel LOAD REPORT from: /Users/pagupta/.cache/huggingface/hub/models--runwayml--stable-diffusion-v1-5/snapshots/451f4fe16113bff5a5d2269ed5ad43b0592e9a14/text_encoder
Key                                | Status     |  | 
---------------

* Running on local URL:  http://127.0.0.1:7863
* To create a public link, set `share=True` in `launch()`.


Generating initial image


100%|██████████| 50/50 [01:14<00:00,  1.49s/it]
Pipelines loaded with `dtype=torch.float16` cannot run with `cpu` device. It is not recommended to move them to `cpu` as running them will fail. Please make sure to use an accelerator to run the pipeline in inference, due to the lack of support for`float16` operations on this device in PyTorch. Please, remove the `torch_dtype=torch.float16` argument, or use another device for inference.
Pipelines loaded with `dtype=torch.float16` cannot run with `cpu` device. It is not recommended to move them to `cpu` as running them will fail. Please make sure to use an accelerator to run the pipeline in inference, due to the lack of support for`float16` operations on this device in PyTorch. Please, remove the `torch_dtype=torch.float16` argument, or use another device for inference.
Pipelines loaded with `dtype=torch.float16` cannot run with `cpu` device. It is not recommended to move them to `cpu` as running them will fail. Please make sure to use an 

Generating video frames


100%|██████████| 25/25 [1:14:02<00:00, 177.69s/it]
Pipelines loaded with `dtype=torch.float16` cannot run with `cpu` device. It is not recommended to move them to `cpu` as running them will fail. Please make sure to use an accelerator to run the pipeline in inference, due to the lack of support for`float16` operations on this device in PyTorch. Please, remove the `torch_dtype=torch.float16` argument, or use another device for inference.
Pipelines loaded with `dtype=torch.float16` cannot run with `cpu` device. It is not recommended to move them to `cpu` as running them will fail. Please make sure to use an accelerator to run the pipeline in inference, due to the lack of support for`float16` operations on this device in PyTorch. Please, remove the `torch_dtype=torch.float16` argument, or use another device for inference.
Pipelines loaded with `dtype=torch.float16` cannot run with `cpu` device. It is not recommended to move them to `cpu` as running them will fail. Please make sure to use 

Saving video


Python(42942) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
Python(42943) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


In [62]:
# !pip install open_clip_torch
import torch

import open_clip
import imageio
from PIL import Image
from torchvision import transforms
import numpy as np

def compute_clipscore(video_path, prompt):

    device = "mps" if torch.backends.mps.is_available() else "cpu"

    # Load CLIP
    #open_clip automatically returns a preprocessing pipeline designed for that CLIP model. For ViT-B-32, the pipeline includes:

    # Resize → 224

    # CenterCrop → 224

    # Convert to tensor

    # Normalize with CLIP mean/std
    
    model, _, preprocess = open_clip.create_model_and_transforms(
        "ViT-B-32", pretrained="openai"
    )
    model = model.to(device)
    model.eval()

    # Tokenize text
    tokenizer = open_clip.get_tokenizer("ViT-B-32")
    text = tokenizer([prompt]).to(device)

    with torch.no_grad():
        text_features = model.encode_text(text)
        text_features = text_features / text_features.norm(dim=-1, keepdim=True)

    # Read video frames
    video = imageio.get_reader(video_path)

    scores = []

    for frame in video:
        image = Image.fromarray(frame)
        image = preprocess(image).unsqueeze(0).to(device)

        with torch.no_grad():
            image_features = model.encode_image(image)
            image_features = image_features / image_features.norm(dim=-1, keepdim=True)

        similarity = (image_features @ text_features.T).item()
        scores.append(similarity)

    video.close()

    return np.mean(scores)


prompt = "cinematic disney style character walking in forest"

video_path = generate_video(prompt)

score = compute_clipscore(video_path, prompt)

print("CLIPScore:", score)
    

Generating initial image


100%|██████████| 50/50 [00:50<00:00,  1.01s/it]
Pipelines loaded with `dtype=torch.float16` cannot run with `cpu` device. It is not recommended to move them to `cpu` as running them will fail. Please make sure to use an accelerator to run the pipeline in inference, due to the lack of support for`float16` operations on this device in PyTorch. Please, remove the `torch_dtype=torch.float16` argument, or use another device for inference.
Pipelines loaded with `dtype=torch.float16` cannot run with `cpu` device. It is not recommended to move them to `cpu` as running them will fail. Please make sure to use an accelerator to run the pipeline in inference, due to the lack of support for`float16` operations on this device in PyTorch. Please, remove the `torch_dtype=torch.float16` argument, or use another device for inference.
Pipelines loaded with `dtype=torch.float16` cannot run with `cpu` device. It is not recommended to move them to `cpu` as running them will fail. Please make sure to use an 

Generating video frames


100%|██████████| 25/25 [15:21<00:00, 36.86s/it]
Pipelines loaded with `dtype=torch.float16` cannot run with `cpu` device. It is not recommended to move them to `cpu` as running them will fail. Please make sure to use an accelerator to run the pipeline in inference, due to the lack of support for`float16` operations on this device in PyTorch. Please, remove the `torch_dtype=torch.float16` argument, or use another device for inference.
Pipelines loaded with `dtype=torch.float16` cannot run with `cpu` device. It is not recommended to move them to `cpu` as running them will fail. Please make sure to use an accelerator to run the pipeline in inference, due to the lack of support for`float16` operations on this device in PyTorch. Please, remove the `torch_dtype=torch.float16` argument, or use another device for inference.
Pipelines loaded with `dtype=torch.float16` cannot run with `cpu` device. It is not recommended to move them to `cpu` as running them will fail. Please make sure to use an 

Saving video


Python(48130) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
Python(48133) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
/Users/pagupta/Desktop/GD_Projects/module_2_all_courses/ML_Specialization_coursera/ml_special_311/lib/python3.11/site-packages/open_clip/factory.py:450: UserWarning: QuickGELU mismatch between final model config (quick_gelu=False) and pretrained tag 'openai' (quick_gelu=True).
  warnings.warn(
Python(48323) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


CLIPScore: 0.2696548067033291


In [59]:
torch.mps.empty_cache()


In [63]:
# !pip install lpips


import lpips
import torch
import imageio
from PIL import Image
import torchvision.transforms as transforms
import numpy as np

def compute_tlpips(video_path):

    device = "mps" if torch.backends.mps.is_available() else "cpu"

    # Load LPIPS model
    loss_fn = lpips.LPIPS(net='alex').to(device)

    video = imageio.get_reader(video_path)

    frames = []
    transform = transforms.Compose([
        transforms.Resize((256,256)),
        transforms.ToTensor(),
        transforms.Normalize([0.5],[0.5])
    ])

    # Load frames
    for frame in video:
        img = Image.fromarray(frame).convert("RGB")
        img = transform(img).unsqueeze(0)
        frames.append(img)

    video.close()

    frames = [f.to(device) for f in frames]

    scores = []

    for i in range(len(frames)-1):
        with torch.no_grad():
            dist = loss_fn(frames[i], frames[i+1])
        scores.append(dist.item())

    return np.mean(scores)

In [64]:
prompt = "cinematic disney style character walking in forest"

In [65]:
# !pip install --upgrade certifi
# import certifi
# print(certifi.where())

# tlpips_score = compute_tlpips(video_path)

# print("CLIPScore:", clip_score)
# print("tLPIPS:", tlpips_score)
import ssl
# Disable SSL certificate verification globally
ssl._create_default_https_context = ssl._create_unverified_context

# ... rest of your code ...
tlpips_score = compute_tlpips(video_path)
print("CLIPScore:", clip_score)
print("tLPIPS:", tlpips_score)


Setting up [LPIPS] perceptual loss: trunk [alex], v[0.1], spatial [off]


/Users/pagupta/Desktop/GD_Projects/module_2_all_courses/ML_Specialization_coursera/ml_special_311/lib/python3.11/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/Users/pagupta/Desktop/GD_Projects/module_2_all_courses/ML_Specialization_coursera/ml_special_311/lib/python3.11/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=AlexNet_Weights.IMAGENET1K_V1`. You can also use `weights=AlexNet_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Loading model from: /Users/pagupta/Desktop/GD_Projects/module_2_all_courses/ML_Specialization_coursera/ml_special_311/lib/python3.11/site-packages/lpips/weights/v0.1/alex.pth


Python(48752) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


CLIPScore: 0.25397221557796
tLPIPS: 0.0591017226378123
